# Understanding Reward Modeling

This notebook demonstrates how reward modeling works for RLHF (Reinforcement Learning from Human Feedback).

## What is Reward Modeling?

Reward modeling is the second phase in RLHF. We train a model to predict human preferences by learning from pairwise comparisons:
- Given two responses to the same prompt
- One is chosen (better), one is rejected (worse)
- The reward model learns to assign higher scores to better responses

## Key Concepts

1. **Preference Pairs**: (prompt, chosen_response, rejected_response)
2. **Bradley-Terry Model**: P(A > B) = sigmoid(R(A) - R(B))
3. **Ranking Loss**: Maximize probability that chosen > rejected
4. **Scalar Rewards**: Model outputs a single number representing quality

Let's see it in action!

In [ ]:
# Setup
import sys
sys.path.append('..')

import torch
from transformers import TrainingArguments
from datasets import Dataset

from src.models.language import LanguageModel
from src.models.reward import RewardModel
from src.data.processors.preference import (
    create_synthetic_preference_data,
    prepare_preference_data,
    PreferenceDataCollator,
)
from src.core.reward_modeling.trainer import RewardModelTrainer, compute_reward_metrics
from src.core.reward_modeling.loss import (
    bradley_terry_loss,
    compute_ranking_accuracy,
    compute_reward_margin,
)
from src.utils.compat import get_training_args_kwargs

## Step 1: Load a Base Model

We'll start with GPT-2. Ideally, this would be an SFT model, but we'll use the base model for demonstration.

In [ ]:
# Load base language model
base_model = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=False,  # Full fine-tuning for reward model
)

print(f"Base model loaded: GPT-2")
print(f"Total parameters: {base_model.num_parameters:,}")

## Step 2: Create Reward Model

Wrap the base model with a reward head that outputs scalar values.

In [ ]:
# Create reward model (adds value head)
reward_model = RewardModel(
    base_model=base_model,
    freeze_base=False,  # Train both base model and value head
)

print(f"\nReward Model Information:")
print(f"  Total parameters: {reward_model.num_parameters:,}")
print(f"  Trainable parameters: {reward_model.num_trainable_parameters:,}")
print(f"  Value head parameters: {reward_model.num_value_head_parameters:,}")
print(f"  Trainable %: {reward_model.percent_trainable:.2f}%")
print(f"  Device: {reward_model.device}")

## Step 3: Understand Preference Data

Preference data consists of triplets: (prompt, chosen, rejected)

Let's create synthetic data to understand the format.

In [ ]:
# Generate synthetic preference data
train_examples = create_synthetic_preference_data(
    num_examples=100,
    seed=42,
)

eval_examples = create_synthetic_preference_data(
    num_examples=20,
    seed=43,
)

print(f"Generated {len(train_examples)} training examples")
print(f"Generated {len(eval_examples)} evaluation examples")

# Show examples
print("\nExample Preference Pairs:")
print("=" * 80)

for i in range(3):
    ex = train_examples[i]
    print(f"\n[Example {i+1}]")
    print(f"Prompt:   {ex['prompt']}")
    print(f"✅ Chosen: {ex['chosen']}")
    print(f"❌ Rejected: {ex['rejected']}")
    print("-" * 80)

## Step 4: Tokenize Preference Pairs

Convert text to token IDs. Each pair becomes two sequences: prompt+chosen and prompt+rejected.

In [ ]:
# Convert to Dataset
train_data = Dataset.from_list(train_examples)
eval_data = Dataset.from_list(eval_examples)

# Tokenize
train_dataset = prepare_preference_data(
    train_data,
    tokenizer=reward_model.tokenizer,
    max_length=128,
    format_fn=None,  # No special format needed for synthetic data
)

eval_dataset = prepare_preference_data(
    eval_data,
    tokenizer=reward_model.tokenizer,
    max_length=128,
    format_fn=None,
)

print(f"\nTokenized {len(train_dataset)} training pairs")
print(f"Tokenized {len(eval_dataset)} evaluation pairs")

# Show tokenized example
example = train_dataset[0]
print("\nExample tokenized pair:")
print(f"  chosen_input_ids: {len(example['chosen_input_ids'])} tokens")
print(f"  rejected_input_ids: {len(example['rejected_input_ids'])} tokens")

## Step 5: Understanding Bradley-Terry Loss

The Bradley-Terry model is the heart of reward modeling:

$$P(\text{chosen} > \text{rejected}) = \sigma(R(\text{chosen}) - R(\text{rejected}))$$

Loss = $-\log(\sigma(R(\text{chosen}) - R(\text{rejected})))$

Let's see it in action!

In [ ]:
# Simulate some rewards
reward_chosen = torch.tensor([2.5, 3.0, 1.5, 4.0])
reward_rejected = torch.tensor([1.0, 0.5, 2.0, 1.5])

# Compute Bradley-Terry loss
loss, details = bradley_terry_loss(
    reward_chosen,
    reward_rejected,
    margin=0.0,
    return_details=True,
)

print("Bradley-Terry Loss Example:")
print(f"\nReward (chosen):   {reward_chosen.tolist()}")
print(f"Reward (rejected): {reward_rejected.tolist()}")
print(f"\nLoss: {loss.item():.4f}")
print(f"Accuracy: {details['accuracy']:.2%}")
print(f"Avg chosen: {details['reward_chosen_mean']:.4f}")
print(f"Avg rejected: {details['reward_rejected_mean']:.4f}")
print(f"Margin: {details['reward_margin_mean']:.4f}")

print("\n💡 Key Insights:")
print("  • Accuracy = 75% (3 out of 4 pairs ranked correctly)")
print("  • Loss decreases as ranking accuracy improves")
print("  • Margin = average separation between chosen and rejected")

## Step 6: Setup Training

Create data collator and training arguments.

In [ ]:
# Data collator (handles padding)
data_collator = PreferenceDataCollator(
    tokenizer=reward_model.tokenizer,
    max_length=128,
)

# Training arguments (version-compatible)
training_args_kwargs = get_training_args_kwargs(
    output_dir="./outputs/reward_tutorial",
    eval_enabled=True,
    logging_dir="./outputs/reward_tutorial/logs",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=20,
    max_grad_norm=1.0,
    logging_steps=5,
    eval_steps=25,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    report_to=[],
    seed=42,
)

training_args = TrainingArguments(**training_args_kwargs)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Evaluation: Every {training_args.eval_steps} steps")

## Step 7: Create Trainer and Train

Use the custom RewardModelTrainer with Bradley-Terry loss.

In [ ]:
# Create reward model trainer
trainer = RewardModelTrainer(
    model=reward_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_reward_metrics,
    margin=0.0,  # No margin for basic training
    log_rewards=True,
    num_rewards_to_log=3,
)

print("Starting reward model training...")
print("Watch for:")
print("  • train/loss: Should decrease")
print("  • train/accuracy: Should increase (target > 70%)")
print("  • train/reward_margin: Should increase (chosen >> rejected)")
print("\n" + "="*60)

# Train!
trainer.train()

print("\n" + "="*60)
print("✅ Training complete!")

## Step 8: Evaluate the Reward Model

Check how well the model ranks preferences.

In [ ]:
# Evaluate
metrics = trainer.evaluate()

print("\nEvaluation Metrics:")
print("=" * 60)

# Key metrics
accuracy = metrics.get('eval_accuracy', 0)
margin = metrics.get('eval_margin_mean', 0)
chosen_mean = metrics.get('eval_chosen_mean', 0)
rejected_mean = metrics.get('eval_rejected_mean', 0)

print(f"\n📊 Ranking Accuracy: {accuracy:.2%}")
print(f"📊 Reward Margin: {margin:.4f}")
print(f"📊 Avg Chosen Reward: {chosen_mean:.4f}")
print(f"📊 Avg Rejected Reward: {rejected_mean:.4f}")

# Performance assessment
print("\n🎯 Performance Assessment:")
if accuracy > 0.75:
    print("  ✅ Excellent! Accuracy > 75%")
    print("  This reward model is ready for RLHF!")
elif accuracy > 0.65:
    print("  ⚠️  Good, but could be better.")
    print("  Consider more training data or longer training.")
elif accuracy > 0.55:
    print("  ⚠️  Fair performance.")
    print("  Needs improvement before RLHF.")
else:
    print("  ❌ Poor performance (barely better than random 50%).")
    print("  Check data quality and model capacity.")

# All metrics
print("\n📋 Full Metrics:")
for key, value in sorted(metrics.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

## Step 9: Test Reward Predictions

Let's manually test the reward model on some examples.

In [ ]:
# Set to eval mode
reward_model.eval()

# Test examples
test_examples = [
    {
        "prompt": "What is 10 + 5?",
        "good": "10 + 5 equals 15.",
        "bad": "I don't know.",
    },
    {
        "prompt": "What is the capital of Italy?",
        "good": "The capital of Italy is Rome.",
        "bad": "Italy has a capital.",
    },
    {
        "prompt": "How do you make coffee?",
        "good": "To make coffee, add water and grounds to machine, then brew.",
        "bad": "You make coffee.",
    },
]

print("Testing Reward Model Predictions:")
print("=" * 80)

with torch.no_grad():
    for i, ex in enumerate(test_examples):
        # Tokenize both responses
        good_text = ex['prompt'] + ex['good']
        bad_text = ex['prompt'] + ex['bad']
        
        good_tokens = reward_model.tokenizer(
            good_text,
            return_tensors="pt",
            max_length=128,
            truncation=True,
        )
        
        bad_tokens = reward_model.tokenizer(
            bad_text,
            return_tensors="pt",
            max_length=128,
            truncation=True,
        )
        
        # Move to device
        good_tokens = {k: v.to(reward_model.device) for k, v in good_tokens.items()}
        bad_tokens = {k: v.to(reward_model.device) for k, v in bad_tokens.items()}
        
        # Get rewards
        reward_good = reward_model(
            input_ids=good_tokens['input_ids'],
            attention_mask=good_tokens['attention_mask'],
            return_dict=False,
        )
        
        reward_bad = reward_model(
            input_ids=bad_tokens['input_ids'],
            attention_mask=bad_tokens['attention_mask'],
            return_dict=False,
        )
        
        reward_good = reward_good.item()
        reward_bad = reward_bad.item()
        margin = reward_good - reward_bad
        
        print(f"\n[Test {i+1}]")
        print(f"Prompt: {ex['prompt']}")
        print(f"\n  ✅ Good Response: {ex['good']}")
        print(f"     Reward: {reward_good:.4f}")
        print(f"\n  ❌ Bad Response: {ex['bad']}")
        print(f"     Reward: {reward_bad:.4f}")
        print(f"\n  📊 Margin: {margin:.4f} {'✓ Correct!' if margin > 0 else '✗ Wrong'}")
        print("-" * 80)

print("\n💡 Key Observations:")
print("  • Good responses should have higher rewards")
print("  • Margin (difference) indicates model confidence")
print("  • Larger margin = more confident ranking")

## Step 10: Visualize Training Progress

Let's plot the training dynamics.

In [ ]:
import matplotlib.pyplot as plt

# Get training metrics
metrics = trainer.get_training_metrics()

if metrics['steps']:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    if metrics['losses']:
        axes[0, 0].plot(metrics['steps'], metrics['losses'], 'b-', linewidth=2)
        axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('Step')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    if metrics['accuracies']:
        axes[0, 1].plot(metrics['steps'], metrics['accuracies'], 'g-', linewidth=2)
        axes[0, 1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random (50%)')
        axes[0, 1].axhline(y=0.7, color='orange', linestyle='--', alpha=0.5, label='Good (70%)')
        axes[0, 1].set_title('Ranking Accuracy', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Step')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # Reward Margin
    if metrics['margins']:
        axes[1, 0].plot(metrics['steps'], metrics['margins'], 'purple', linewidth=2)
        axes[1, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[1, 0].set_title('Reward Margin (Chosen - Rejected)', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Step')
        axes[1, 0].set_ylabel('Margin')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Reward Distribution
    if metrics['reward_chosen_means'] and metrics['reward_rejected_means']:
        axes[1, 1].plot(metrics['steps'], metrics['reward_chosen_means'], 'g-', 
                       linewidth=2, label='Chosen')
        axes[1, 1].plot(metrics['steps'], metrics['reward_rejected_means'], 'r-', 
                       linewidth=2, label='Rejected')
        axes[1, 1].set_title('Mean Rewards', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Step')
        axes[1, 1].set_ylabel('Reward')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot")

print("\n📈 What to Look For:")
print("  • Loss: Should decrease steadily")
print("  • Accuracy: Should rise above 50% (random) toward 70-80%")
print("  • Margin: Should increase (stronger separation)")
print("  • Rewards: Chosen should stay above rejected")

## Step 11: Save the Reward Model

Save for use in RLHF (Phase 5).

In [ ]:
# Save model
save_path = "./outputs/reward_tutorial/final_model"
reward_model.save_pretrained(save_path)

print(f"✅ Reward model saved to: {save_path}")
print("\nYou can load it later with:")
print(f'reward_model = RewardModel.from_pretrained("{save_path}")')

## Summary

In this notebook, we:
1. ✅ Loaded a base language model (GPT-2)
2. ✅ Created a reward model with value head
3. ✅ Generated synthetic preference data
4. ✅ Understood Bradley-Terry ranking loss
5. ✅ Trained the reward model
6. ✅ Evaluated ranking accuracy
7. ✅ Tested reward predictions manually
8. ✅ Visualized training dynamics
9. ✅ Saved the model for RLHF

## Key Takeaways

- **Reward models predict scalar values**: Higher = better response
- **Bradley-Terry loss**: Maximizes P(chosen > rejected)
- **Ranking accuracy**: Primary metric (target > 70%)
- **Margin matters**: Larger separation = more confident
- **Foundation for RLHF**: This reward model guides RL training

## Implemented Features

✅ **LoRA for memory efficiency** (see Advanced 5)
✅ **Training on SFT model** (see Advanced 6)
✅ **SFT + LoRA best practice** (see Advanced 7)

## Next Steps

- Try with real datasets (Anthropic HH-RLHF) - see Advanced 1
- Experiment with frozen base model (faster training) - see Advanced 2
- Apply margin-based loss for stronger separation - see Advanced 3
- Check calibration for production models - see Advanced 4
- Move to Phase 4: DPO or Phase 5: PPO/RLHF!

---

# Advanced Topics

Let's explore some advanced reward modeling techniques!

## Advanced 1: Training with Real Data (Anthropic HH-RLHF)

Let's load and process real human preference data.

In [ ]:
from datasets import load_dataset
from src.data.processors.preference import parse_anthropic_format, load_preference_dataset

# Note: This requires internet connection and may take time
# Uncomment to try:

# # Load Anthropic HH-RLHF dataset
# print("Loading Anthropic HH-RLHF dataset...")
# real_train = load_preference_dataset(
#     dataset_name="Anthropic/hh-rlhf",
#     split="train",
# )
# 
# real_test = load_preference_dataset(
#     dataset_name="Anthropic/hh-rlhf",
#     split="test",
# )
# 
# print(f"Train: {len(real_train)} examples")
# print(f"Test: {len(real_test)} examples")
# 
# # Show example
# example = real_train[0]
# print("\nRaw example (conversational format):")
# print(f"Chosen: {example['chosen'][:200]}...")
# 
# # Parse Anthropic format
# parsed = parse_anthropic_format(example)
# print("\nParsed example:")
# print(f"Prompt: {parsed['prompt'][:100]}...")
# print(f"Chosen: {parsed['chosen'][:100]}...")
# print(f"Rejected: {parsed['rejected'][:100]}...")
# 
# # Prepare for training
# real_train_dataset = prepare_preference_data(
#     real_train.select(range(1000)),  # Use subset
#     tokenizer=reward_model.tokenizer,
#     max_length=512,
#     format_fn=parse_anthropic_format,
# )
# 
# print(f"\n✅ Prepared {len(real_train_dataset)} examples for training")

print("\n💡 Tips for Real Data:")
print("  • Use format_fn=parse_anthropic_format for conversational data")
print("  • Increase max_length to 512 or 1024 for longer conversations")
print("  • Start with a subset (1K-10K) for faster iteration")
print("  • Real data is noisier than synthetic (expect lower accuracy)")

## Advanced 2: Frozen Base Model (Train Only Value Head)

For faster training, freeze the base model and only train the value head.

In [ ]:
# Create reward model with frozen base
frozen_reward_model = RewardModel(
    base_model=base_model,
    freeze_base=True,  # Freeze all base model parameters
)

print("Frozen Base Model Configuration:")
print(f"  Total parameters: {frozen_reward_model.num_parameters:,}")
print(f"  Trainable parameters: {frozen_reward_model.num_trainable_parameters:,}")
print(f"  Trainable %: {frozen_reward_model.percent_trainable:.4f}%")

print("\n⚡ Benefits:")
print("  • 10-100x faster training")
print("  • Much lower memory usage")
print("  • Still effective for reward modeling")

print("\n⚠️  Trade-offs:")
print("  • Lower capacity (only head adapts)")
print("  • Works best with strong SFT base model")
print("  • May need more data for good performance")

## Advanced 3: Margin-Based Loss

Add a margin to enforce stronger separation between chosen and rejected.

In [ ]:
# Compare different margins
test_chosen = torch.tensor([3.0, 2.5, 4.0])
test_rejected = torch.tensor([2.0, 2.0, 1.0])

margins = [0.0, 0.5, 1.0, 2.0]

print("Effect of Margin Parameter:")
print("=" * 60)
print(f"\nReward chosen:   {test_chosen.tolist()}")
print(f"Reward rejected: {test_rejected.tolist()}")
print(f"Natural margin:  {(test_chosen - test_rejected).tolist()}")

print("\nLoss with different margins:")
for margin in margins:
    loss, details = bradley_terry_loss(
        test_chosen,
        test_rejected,
        margin=margin,
        return_details=True,
    )
    print(f"  Margin = {margin}: Loss = {loss.item():.4f}, Acc = {details['accuracy']:.2%}")

print("\n💡 Interpretation:")
print("  • margin=0.0: Standard Bradley-Terry (default)")
print("  • margin>0: Requires larger separation")
print("  • Higher margin = higher loss (stronger training signal)")
print("  • Use margin=0.5-1.0 for more confident rankings")

## Advanced 4: Understanding Reward Calibration

Calibration measures whether the model's confidence matches actual accuracy.

In [ ]:
from src.core.reward_modeling.loss import calibration_error

# Simulate predictions
# Well-calibrated: high confidence → high accuracy
n = 100
chosen_calibrated = torch.randn(n) + 1.0  # Biased higher
rejected_calibrated = torch.randn(n)

# Poorly calibrated: overconfident
chosen_poor = torch.randn(n) + 0.2  # Small actual difference
rejected_poor = torch.randn(n)

# Compute calibration
ece_good = calibration_error(chosen_calibrated, rejected_calibrated, num_bins=10)
ece_poor = calibration_error(chosen_poor, rejected_poor, num_bins=10)

print("Calibration Analysis:")
print("=" * 60)
print(f"\nWell-calibrated model:")
print(f"  Accuracy: {compute_ranking_accuracy(chosen_calibrated, rejected_calibrated):.2%}")
print(f"  ECE (Expected Calibration Error): {ece_good:.4f}")

print(f"\nPoorly-calibrated model:")
print(f"  Accuracy: {compute_ranking_accuracy(chosen_poor, rejected_poor):.2%}")
print(f"  ECE: {ece_poor:.4f}")

print("\n💡 Calibration Insights:")
print("  • ECE = 0: Perfect calibration")
print("  • ECE < 0.1: Well calibrated")
print("  • ECE > 0.2: Poorly calibrated (overconfident or underconfident)")
print("  • Important for RLHF: Well-calibrated rewards → stable training")

## Advanced 5: Using LoRA for Memory Efficiency

LoRA (Low-Rank Adaptation) dramatically reduces memory usage and training time by only updating small adapter matrices instead of the full model.

**Memory Savings**: For GPT-2 (124M params), LoRA reduces trainable parameters by ~50x!

In [ ]:
# Load base model with LoRA
lora_base_model = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,
    lora_config={
        "r": 8,  # Rank of adapter matrices
        "lora_alpha": 16,  # Scaling factor
        "lora_dropout": 0.05,
        "target_modules": ["c_attn", "c_proj"],  # Which layers to adapt
        "bias": "none",
        "task_type": "CAUSAL_LM",
    },
)

print("Base Model with LoRA:")
print(f"  Total parameters: {lora_base_model.num_parameters:,}")
print(f"  Trainable parameters: {lora_base_model.num_trainable_parameters:,}")
print(f"  Trainable %: {(lora_base_model.num_trainable_parameters / lora_base_model.num_parameters * 100):.4f}%")

# Create reward model from LoRA base
lora_reward_model = RewardModel(
    base_model=lora_base_model,
    freeze_base=False,  # Train LoRA adapters + value head
)

print(f"\nReward Model with LoRA:")
print(f"  Total parameters: {lora_reward_model.num_parameters:,}")
print(f"  Trainable parameters: {lora_reward_model.num_trainable_parameters:,}")
print(f"  Value head parameters: {lora_reward_model.num_value_head_parameters:,}")
print(f"  Trainable %: {lora_reward_model.percent_trainable:.2f}%")

print("\n🎯 Benefits of LoRA:")
print(f"  • Only training {lora_reward_model.num_trainable_parameters:,} parameters")
print(f"  • ~50x fewer parameters than full fine-tuning!")
print(f"  • Faster training (less gradient computation)")
print(f"  • Lower memory (smaller optimizer states)")
print(f"  • Easy to save/load (only adapter weights)")

print("\n⚠️  When to Use LoRA:")
print("  ✅ Limited GPU memory (< 16GB)")
print("  ✅ Want faster iteration")
print("  ✅ Base model already strong (GPT-2, LLaMA)")
print("  ❌ Very small models (< 100M params, overhead dominates)")
print("  ❌ Need maximum capacity (full fine-tuning slightly better)")

# Optional: Train this model (same code as before)
# Just replace 'reward_model' with 'lora_reward_model' in trainer

## Advanced 6: Training on SFT Model Instead of Base Model

**Best Practice**: Always start reward modeling from an SFT (Supervised Fine-Tuned) model, not the base model!

**Why?**
- SFT model already produces good responses (just need to rank them)
- Base model may produce incoherent text (hard to learn ranking)
- Faster convergence (fewer training steps needed)
- Better final performance (higher ranking accuracy)

Let's see how to load and use an SFT checkpoint.

In [ ]:
# Example 1: Load from local SFT checkpoint
# If you trained an SFT model in Phase 2, load it here

# Uncomment if you have a local SFT checkpoint:
# sft_model = LanguageModel.from_pretrained(
#     "./outputs/sft_tutorial/final_model",  # Path to your SFT checkpoint
#     use_lora=False,  # Or True if SFT used LoRA
# )

# Example 2: Load from HuggingFace Hub
# Many instruction-tuned models are available:
# - "microsoft/phi-2" (2.7B, instruction-tuned)
# - "TinyLlama/TinyLlama-1.1B-Chat-v1.0" (1.1B, chat-tuned)
# - Or your own uploaded SFT model

# For this demo, we'll simulate with GPT-2 (pretend it's SFT)
print("Loading SFT Model...")
print("📝 In production, replace 'gpt2' with your actual SFT checkpoint path")
print("   Example: './outputs/sft_gpt2/final_model' or 'your-org/gpt2-sft-alpaca'")
print()

# Load the "SFT" model (in production, use your actual SFT checkpoint)
sft_model = LanguageModel.from_pretrained(
    "gpt2",  # Replace with your SFT model path
    use_lora=False,
)

print("✅ SFT Model Loaded")
print(f"  Model: gpt2 (replace with your SFT model)")
print(f"  Parameters: {sft_model.num_parameters:,}")
print()

# Create reward model from SFT base
sft_reward_model = RewardModel(
    base_model=sft_model,
    freeze_base=False,  # Train both base and value head
)

print("Reward Model from SFT:")
print(f"  Total parameters: {sft_reward_model.num_parameters:,}")
print(f"  Trainable parameters: {sft_reward_model.num_trainable_parameters:,}")
print(f"  Value head: {sft_reward_model.num_value_head_parameters:,}")

print("\n🎯 Why Start from SFT Model:")
print("  ✅ SFT model already instruction-following")
print("  ✅ Produces coherent, helpful responses")
print("  ✅ Reward model only needs to learn preference ranking")
print("  ✅ Faster training (2-3 epochs vs 5-10 epochs)")
print("  ✅ Better accuracy (75-85% vs 65-75%)")

print("\n📊 Expected Performance Comparison:")
print("  Base Model → Reward Model:")
print("    • Accuracy: 65-75%")
print("    • Training time: 5-10 epochs")
print("    • Convergence: Slower")
print()
print("  SFT Model → Reward Model:")
print("    • Accuracy: 75-85%")
print("    • Training time: 1-3 epochs")
print("    • Convergence: Faster")

print("\n💡 Complete Workflow:")
print("  1. Phase 2: Train SFT model on instructions")
print("  2. Phase 3: Load SFT → Train reward model on preferences")
print("  3. Phase 5: Use reward model for PPO/RLHF")

print("\n🔧 To use this in training:")
print("  # Just replace 'reward_model' with 'sft_reward_model'")
print("  trainer = RewardModelTrainer(")
print("      model=sft_reward_model,  # ← Use SFT-based reward model")
print("      args=training_args,")
print("      train_dataset=train_dataset,")
print("      eval_dataset=eval_dataset,")
print("      data_collator=data_collator,")
print("  )")
print("  trainer.train()")

## Advanced 7: Combining LoRA + SFT (Best Practice)

**Production Recommendation**: Use LoRA on an SFT model for the best balance of performance, speed, and memory efficiency!

In [ ]:
# Best Practice: Load SFT model with LoRA
print("🏆 Production Best Practice: SFT Model + LoRA")
print("=" * 60)

# Load SFT model with LoRA adapters
sft_lora_model = LanguageModel.from_pretrained(
    "gpt2",  # Replace with your SFT checkpoint
    use_lora=True,
    lora_config={
        "r": 16,  # Slightly higher rank for reward modeling
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": ["c_attn", "c_proj"],
        "bias": "none",
        "task_type": "CAUSAL_LM",
    },
)

print("\n✅ SFT Model with LoRA Loaded")
print(f"  Total parameters: {sft_lora_model.num_parameters:,}")
print(f"  Trainable (LoRA): {sft_lora_model.num_trainable_parameters:,}")
print(f"  Trainable %: {(sft_lora_model.num_trainable_parameters / sft_lora_model.num_parameters * 100):.4f}%")

# Create reward model
best_practice_reward_model = RewardModel(
    base_model=sft_lora_model,
    freeze_base=False,  # Train LoRA adapters + value head
)

print(f"\nReward Model (SFT + LoRA):")
print(f"  Total parameters: {best_practice_reward_model.num_parameters:,}")
print(f"  Trainable parameters: {best_practice_reward_model.num_trainable_parameters:,}")
print(f"  LoRA adapters: {sft_lora_model.num_trainable_parameters:,}")
print(f"  Value head: {best_practice_reward_model.num_value_head_parameters:,}")
print(f"  Trainable %: {best_practice_reward_model.percent_trainable:.2f}%")

print("\n🎯 Why This is Best Practice:")
print("  ✅ Starts from high-quality SFT base (better responses)")
print("  ✅ Uses LoRA (50x fewer trainable parameters)")
print("  ✅ Fast training (< 1 hour on single GPU)")
print("  ✅ Low memory (fits on 8-12GB GPU)")
print("  ✅ High accuracy (75-85% ranking)")
print("  ✅ Easy to save/load (small checkpoint)")

print("\n📊 Resource Comparison:")
print()
print("  Approach                 | Trainable Params | Memory | Time | Accuracy")
print("  " + "-" * 75)
print("  Base + Full Fine-tune    | ~124M (100%)     | 16GB+  | 8hr  | 65-75%")
print("  Base + LoRA              | ~2M (2%)         | 4GB    | 2hr  | 65-75%")
print("  SFT + Full Fine-tune     | ~124M (100%)     | 16GB+  | 4hr  | 75-85%")
print("  SFT + LoRA ⭐            | ~2M (2%)         | 4GB    | 1hr  | 75-85%")
print()
print("  ⭐ = Recommended for most use cases")

print("\n💡 When to Use Each Approach:")
print()
print("  1. Base + Full Fine-tune:")
print("     • Research/academic settings")
print("     • Unlimited compute budget")
print("     • Want absolute maximum capacity")
print()
print("  2. Base + LoRA:")
print("     • No SFT model available")
print("     • Limited GPU memory")
print("     • Quick experiments")
print()
print("  3. SFT + Full Fine-tune:")
print("     • Production with large GPU cluster")
print("     • Need highest possible accuracy")
print("     • Budget for compute")
print()
print("  4. SFT + LoRA (THIS!):")
print("     • Most production use cases")
print("     • Single GPU (8-16GB)")
print("     • Fast iteration needed")
print("     • Good accuracy required (75-85%)")

print("\n🚀 Ready to Train!")
print("  Use 'best_practice_reward_model' in your trainer:")
print()
print("  trainer = RewardModelTrainer(")
print("      model=best_practice_reward_model,")
print("      args=training_args,")
print("      train_dataset=train_dataset,")
print("      eval_dataset=eval_dataset,")
print("      data_collator=data_collator,")
print("  )")
print("  trainer.train()")
print()
print("  Expected: ~75-85% accuracy after 1-3 epochs!")

## Final Summary: Production Checklist

### ✅ Best Practices for Reward Modeling

1. **Data**:
   - Use 10K-100K high-quality preference pairs
   - Ensure diversity in prompts and responses
   - Validate data quality (check for inconsistencies)
   - See Advanced 1 for real data (Anthropic HH-RLHF)

2. **Model** (⭐ = Recommended):
   - ⭐ **Best Practice**: Start with SFT model + LoRA (see Advanced 7)
   - Alternative: SFT + full fine-tuning (higher accuracy, more compute)
   - Budget option: Base model + LoRA (see Advanced 5)
   - See Advanced 6 for detailed SFT model workflow

3. **Training**:
   - Learning rate: 1e-5 to 5e-5
   - Epochs: 1-3 (often 1 is enough with SFT base)
   - Batch size: 4-8 (process pairs, needs memory)
   - Margin: 0.0 (standard) or 0.5-1.0 (stronger separation) - see Advanced 3
   - Try frozen base first for faster iteration - see Advanced 2

4. **Evaluation**:
   - Target ranking accuracy: > 70% (SFT base: 75-85%)
   - Monitor margin: Should increase during training
   - Check calibration: ECE < 0.1 preferred - see Advanced 4

5. **Before RLHF**:
   - Verify accuracy > 70% on held-out test set
   - Test on diverse examples manually
   - Ensure rewards are reasonable (not too extreme)
   - Calibrated model → more stable RLHF training

### 📚 Advanced Topics Summary

- **Advanced 1**: Real data (Anthropic HH-RLHF)
- **Advanced 2**: Frozen base model (10-100x faster)
- **Advanced 3**: Margin-based loss (stronger separation)
- **Advanced 4**: Calibration analysis (confidence vs accuracy)
- **Advanced 5**: LoRA for memory efficiency (50x fewer params)
- **Advanced 6**: Training on SFT model (75-85% accuracy)
- **Advanced 7**: SFT + LoRA best practice (optimal trade-off)

### 🏆 Production Recommendation

**For most use cases, use Advanced 7 (SFT + LoRA)**:
- Load your SFT checkpoint with LoRA enabled
- Train reward model (LoRA adapters + value head)
- Expected: 75-85% accuracy in 1-3 epochs
- Fits on 8-12GB GPU, trains in < 1 hour

---

## What's Next?

Now you understand reward modeling completely! You're ready for:

- **Phase 4: DPO** (notebooks/04_dpo_simplified_rlhf.ipynb) - Train policy directly from preferences (simpler than RLHF)
- **Phase 5: PPO/RLHF** - Full RL pipeline with this reward model

See you in the next phase! 🚀